In [0]:
SELECT *
FROM bright_.coffee.bright_coffee_shop
LIMIT 10;


-- How big is the data?

SELECT COUNT(*) AS number_of_rows
FROM bright_.coffee.bright_coffee_shop; -- Answer: 149116 but the system is only showing 1000 rows

-- Check for duplicates:

SELECT COUNT(DISTINCT transaction_id)
from bright_.coffee.bright_coffee_shop; -- Answer: 149116. No duplicates because the number of rows= number of transaction_ids regardless of the system only displaying 1000 rows.

-- Is there NULL or an empty space in the dataset?
SELECT *
FROM bright_.coffee.bright_coffee_shop
WHERE transaction_id IS NULL; -- no rows were returned
-- You have to check a specific column to check for null spaces


-- What is the duration of the dataset recorded?

SELECT DISTINCT transaction_date
from bright_.coffee.bright_coffee_shop; -- The duration of the reorded transactions is 6 months (January to June 2023)
-- Exclude the transaction time from the analysis because it is recording the real-time (2026-07-13)

-- Store_ location analysis
SELECT DISTINCT store_location
FROM bright_.coffee.bright_coffee_shop; -- We have 3 store locations

-- Product analysis

SELECT DISTINCT product_category
FROM bright_.coffee.bright_coffee_shop; -- 9 product categories

SELECT DISTINCT product_type
FROM FROM bright_.coffee.bright_coffee_shop; -- 29 different product types, I wanna group the 'Brewed Black tea,Brewed green tea, and Brewed herbal tea,Brewed chai tea' to be one product type, which is 'Brewed tea' using case statement.

SELECT DISTINCT
    CASE
        WHEN product_type IN ('Brewed Black tea','Brewed Green tea','Brewed herbal tea','Brewed Chai tea') THEN 'Brewed tea'
        ELSE product_type
        END AS sub_category
FROM bright_.coffee.bright_coffee_shop; -- 26 different product types


SELECT DISTINCT product_detail
FROM bright_.coffee.bright_coffee_shop; -- 80 diffrerent product_detail, will not interfere with this column as it is unique to the order.

-- Sales Analysis from the transaction_date

SELECT transaction_date,
       DAY(transaction_date) AS day_of_sale,
       MONTHNAME(transaction_date) AS mnth_name,
       DAYNAME(transaction_date) AS dayname_of_sale,

    CASE
        WHEN dayname_of_sale IN ('Sun','Sat') THEN 'weekend'
        ELSE 'weekday'
        END AS day_class

FROM bright_.coffee.bright_coffee_shop;


-- WHich product has the brings the highest revenue?

-- First calculate the revenue for each row/ each sale transaction

SELECT *,
        (unit_price*transaction_qty) AS revenue
FROM  bright_.coffee.bright_coffee_shop;

-- 2nd find the total revenue by store_location
SELECT store_location,
       SUM(unit_price*transaction_qty) AS total_revenue
FROM bright_.coffee.bright_coffee_shop
GROUP BY store_location
ORDER BY total_revenue DESC;

-- 3rd determine which product category brings the highest revenue
SELECT 
       product_category,
       SUM(unit_price*transaction_qty) AS total_revenue
FROM bright_.coffee.bright_coffee_shop
GROUP BY product_category
ORDER BY total_revenue DESC;

-- FINALLY determine total revenue by store_location and product_category
SELECT store_location,
       product_category,
       SUM(unit_price*transaction_qty) AS total_revenue
FROM bright_.coffee.bright_coffee_shop
GROUP BY product_category,
         store_location
ORDER BY 
total_revenue DESC;

WITH Cte1 AS(
SELECT DISTINCT  transaction_id,
                 transaction_date,
       DAY(transaction_date) AS day_of_sale,
       MONTHNAME(transaction_date) AS mnth_name,
       DAYNAME(transaction_date) AS dayname_of_sale,

    CASE
        WHEN DAYNAME(transaction_date) IN ('Sun','Sat') THEN 'weekend'
        ELSE 'weekday'
        END AS day_class,
        
        store_location,
        product_category,

    CASE
        WHEN product_type IN ('Brewed Black tea','Brewed Green tea','Brewed herbal tea','Brewed Chai tea') THEN 'Brewed tea'
        ELSE product_type
        END AS sub_category,
        product_detail,
        transaction_qty,
        (unit_price*transaction_qty) AS revenue

FROM  bright_.coffee.bright_coffee_shop
                
)
-- Error in running because of the alias (dayname_of_sale) cannot be used in the case statement and in GROUP BY ALL statement thereof. 
-- Coorection: when building a CTE rather edit your day_class case statement to be 'WHEN DAYNAME(transaction_date...'
-- Since I did not use any aggregate function in the select statement, the 'GROUP BY' statement is not necessary at all, so I will remove it.

----------- END OF ANALYSIS-----------
       